# Assignment: Flow Matching for Audio Generation

In this assignment you will implement the **inference and training** code for a pitch-conditioned
flow matching model trained on the [NSynth](https://magenta.tensorflow.org/datasets/nsynth)
dataset. A pretrained model (`pretrained_keyboard.pt`) is provided so you can hear results
immediately and focus on understanding the algorithms.

**Before you begin:** Runtime → Change runtime type → **T4 GPU** (required).

---

## What is given to you
| File | Contents |
|---|---|
| `dataset.py` | Spectrogram extraction, normalization, `NSynthSpecDataset` |
| `model.py` | Three model architectures and `build_model_from_config` |
| `pretrained_keyboard.pt` | Pretrained UNet-based flow model (~125k params, keyboard sounds) |

You will **not** use `train.py`, `eval.py`, or `infer.py` — you write equivalent code yourself.

---

## Parts and points
| Part | Task | Points |
|---|---|---|
| 1 | Euler sampling | 20 |
| 2a | Naive velocity scaling (warmup) | 10 |
| 2b | Classifier-Free Guidance (CFG) | 20 |
| 3a | Heun's method | 20 |
| 3b | RK4 (bonus) | 5 |
| 4 | Fine-tuning on a new instrument | 20 |
| 5 | Beat the baseline (open-ended) | 10 |
| **Total** | | **100 + 5 bonus** |

---

## Grading
- **Parts 1–3**: autograder runs your functions on fixed test inputs and checks outputs.
- **Part 4**: autograder (a) tests `train_step` runs 1 gradient step correctly,
  (b) verifies your submitted samples are reproducible from submitted noise seeds.
- **Part 5**: FD / pitch accuracy computed on your submitted samples vs. the baseline.

## Setup

In [ ]:
# Install missing deps (torch/torchaudio are pre-installed on Colab)
!pip install -q librosa tqdm

import os, sys, json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader
import IPython.display as ipd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Clone the repo  ← fill in your repo URL before distributing to students
REPO_URL = 'https://github.com/YOUR_USERNAME/nsynth-speedrun'

if not os.path.exists('/content/nsynth-speedrun'):
    !git clone {REPO_URL} /content/nsynth-speedrun

%cd /content/nsynth-speedrun
!ls

In [ ]:
# Load dataset and model utilities
from dataset import NSynthSpecDataset, wav_to_spec, spec_to_audio, FREQ_BINS, TIME_FRAMES, SR
from model  import build_model_from_config, NULL_PITCH

# Load the pretrained model
CKPT_PATH = '/content/nsynth-speedrun/pretrained_keyboard.pt'
ckpt  = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model = build_model_from_config(ckpt['config']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'Model: {ckpt["n_params"]:,} parameters  ({ckpt["n_params"]/1e3:.0f}k)')
print(f'Input shape : (batch, 2, {FREQ_BINS}, {TIME_FRAMES})')
print(f'  2 channels = real + imaginary parts of a 0.5-second complex STFT')
print(f'  freq_bins  = {FREQ_BINS} = n_fft/2 + 1  (n_fft=256)')
print(f'  time_frames= {TIME_FRAMES}')
print(f'NULL_PITCH  : {NULL_PITCH}  (the "no conditioning" token used for CFG)')
print(f'Sample rate : {SR} Hz')

---
## Background: OT-CFM in one page

**Training objective** (Optimal-Transport Conditional Flow Matching):

Given a data sample $x_1$ (spectrogram), noise $x_0 \sim \mathcal{N}(0,I)$, and pitch label $p$:
$$x_t = (1-t)\,x_0 + t\,x_1 \qquad t \sim U[0,1]$$
$$v^* = x_1 - x_0 \quad\text{(constant velocity along the straight path)}$$
$$\mathcal{L} = \mathbb{E}_{t,x_0,x_1}\bigl[\|\theta(x_t, t, p) - v^*\|^2\bigr]$$

The model $\theta(x_t, t, p)$ predicts the velocity that would carry $x_0$ straight to $x_1$.

**Inference** — integrate the ODE $\dot{x} = \theta(x_t, t, p)$ from $t=0$ to $t=1$:
$$x_{t+\Delta t} = x_t + \theta(x_t,\, t,\, p)\,\Delta t$$

**Classifier-Free Guidance (CFG)** — amplify the conditioning signal:
$$v = v_{\text{uncond}} + s\,(v_{\text{cond}} - v_{\text{uncond}})$$
where $v_{\text{uncond}} = \theta(x_t,\,t,\,\texttt{NULL\_PITCH})$. When $s>1$ the model is pushed
further in the direction that distinguishes the conditioned from the unconditioned output.

**Key model interface:**
```python
v = model(x, t, pitch)
# x     : (B, 2, FREQ_BINS, TIME_FRAMES)  — current noisy spectrogram
# t     : (B,)  float in [0, 1]           — current time
# pitch : (B,)  int  in [0, 127]          — MIDI pitch (or NULL_PITCH=128 for uncond)
# v     : same shape as x                 — predicted velocity
```

---
## Part 1 — Euler Sampling  `[20 pts]`

Implement the basic Euler ODE solver. Starting from Gaussian noise, take `n_steps` equal steps
of size $\Delta t = 1/n$ toward the data distribution:
$$x_{t+\Delta t} = x_t + \theta(x_t,\, t,\, p)\,\Delta t, \qquad t = 0, \Delta t, 2\Delta t, \ldots$$

**Hints:**
- All tensors are already on the right device — don't call `.to()` inside the loop.
- Create the time tensor `t_batch = torch.full((B,), t_val, device=x.device)` before each model call.
- Wrap the entire function body in `torch.no_grad()` for efficiency.
- The last Euler step starts at $t = (n-1)/n$ and lands at $t = 1$.

In [ ]:
def euler_sample(model, x0, pitches, n_steps=50):
    """
    Euler ODE integration from noise to data.

    Parameters
    ----------
    model   : flow model (eval mode, on device)
    x0      : (B, 2, FREQ_BINS, TIME_FRAMES)  initial Gaussian noise
    pitches : (B,) MIDI pitches, dtype=torch.long
    n_steps : number of Euler steps

    Returns
    -------
    x1 : (B, 2, FREQ_BINS, TIME_FRAMES)  generated spectrograms
    """
    ### YOUR CODE HERE ###
    raise NotImplementedError

In [ ]:
# === Autograder Test 1 — do not modify ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out1 = euler_sample(model, x0_ag.clone(), p_ag, n_steps=20)

assert out1.shape == (4, 2, FREQ_BINS, TIME_FRAMES), \
    f'Wrong output shape: {out1.shape}'
assert not torch.allclose(out1, x0_ag, atol=1e-3), \
    'Output equals input — did you implement the integration loop?'
assert out1.isfinite().all(), 'Output contains NaN or Inf'
assert out1.std() > 0.05, f'Output std={out1.std():.4f} is suspiciously low'
print(f'\u2713 euler_sample | shape={out1.shape}, mean={out1.mean():.4f}, std={out1.std():.4f}')

In [ ]:
# Listen: C major scale (C4 D4 E4 F4 G4 A4 B4 C5)
NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

torch.manual_seed(7)
demo_pitches = torch.tensor([60,62,64,65,67,69,71,72], dtype=torch.long, device=device)
demo_noise   = torch.randn(len(demo_pitches), 2, FREQ_BINS, TIME_FRAMES, device=device)

samples_euler = euler_sample(model, demo_noise.clone(), demo_pitches, n_steps=50)

print('Euler samples (no CFG):  notice how pitch identity may be weak at scale=1\n')
for spec, pitch in zip(samples_euler, demo_pitches.tolist()):
    audio = spec_to_audio(spec.cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    name  = NOTE_NAMES[pitch % 12] + str(pitch // 12 - 1)
    print(f'  {name} (MIDI {pitch})')
    display(ipd.Audio(audio.numpy(), rate=SR))

---
## Part 2a — Naive Velocity Scaling  `[10 pts]`

Before implementing proper CFG, let's explore a **wrong but instructive** idea:
just multiply the velocity by a scalar `scale` at each step.

$$x_{t+\Delta t} = x_t + \underbrace{\theta(x_t, t, p) \cdot \texttt{scale}}_{\text{scaled velocity}} \cdot \Delta t$$

When `scale=1.0` this should be identical to `euler_sample`.  
When `scale>1` the integration "goes faster" — think about what happens when you overshoot.

Try `scale=2.0` and `scale=0.5` and listen to the results. What do you notice?

In [ ]:
def naive_scale_sample(model, x0, pitches, n_steps=50, scale=1.0):
    """
    Euler sampling with velocity multiplied by `scale`.

    Parameters: same as euler_sample, plus
    scale : float — multiply every velocity prediction by this factor

    Returns: (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    ### YOUR CODE HERE ###
    raise NotImplementedError

In [ ]:
# === Autograder Test 2a — do not modify ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_s1 = naive_scale_sample(model, x0_ag.clone(), p_ag, n_steps=20, scale=1.0)
out_e  = euler_sample(      model, x0_ag.clone(), p_ag, n_steps=20)
out_s2 = naive_scale_sample(model, x0_ag.clone(), p_ag, n_steps=20, scale=2.0)

assert torch.allclose(out_s1, out_e, atol=1e-5), \
    'naive_scale_sample(scale=1.0) must match euler_sample exactly'
assert not torch.allclose(out_s2, out_e, atol=1e-3), \
    'naive_scale_sample(scale=2.0) should differ from scale=1.0'
print('\u2713 naive_scale_sample | scale=1.0 matches Euler, scale=2.0 differs')

In [ ]:
# Compare scale=1.0 vs scale=2.0 — what does scaling velocity actually do?
torch.manual_seed(42)
test_pitch  = torch.full((1,), 60, dtype=torch.long, device=device)
test_noise  = torch.randn(1, 2, FREQ_BINS, TIME_FRAMES, device=device)

for scale in [0.5, 1.0, 2.0, 4.0]:
    s = naive_scale_sample(model, test_noise.clone(), test_pitch, n_steps=50, scale=scale)
    audio = spec_to_audio(s[0].cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    print(f'scale={scale}')
    display(ipd.Audio(audio.numpy(), rate=SR))

---
## Part 2b — Classifier-Free Guidance (CFG)  `[20 pts]`

The correct way to amplify conditioning is to run the model **twice** per step:
once with the actual pitch and once with `NULL_PITCH` (the "no conditioning" token),
then combine:

$$v_{\text{cond}}   = \theta(x_t,\, t,\, p)\\ v_{\text{uncond}} = \theta(x_t,\, t,\, \texttt{NULL\_PITCH})\\ v = v_{\text{uncond}} + s \cdot (v_{\text{cond}} - v_{\text{uncond}})$$

Notice: at $s=1$, this equals $v_{\text{cond}}$ (standard Euler). At $s=0$ it equals $v_{\text{uncond}}$.
Values $s>1$ extrapolate **beyond** the conditional estimate, sharpening pitch adherence.

**Why is this better than naive scaling?**  
Naive scaling changes the *magnitude* of the velocity (affects "how far" you step).  
CFG changes the *direction* — it steers toward regions that are uniquely associated
with the conditioned pitch, without just overshooting.

**Hints:**
- `torch.full_like(pitches, NULL_PITCH)` creates the unconditional pitch batch.
- When `guidance_scale == 1.0`, skip the second model call (just return `v_cond`).

In [ ]:
def cfg_sample(model, x0, pitches, n_steps=50, guidance_scale=1.0):
    """
    Euler sampling with Classifier-Free Guidance.

    Parameters
    ----------
    model          : flow model
    x0             : (B, 2, FREQ_BINS, TIME_FRAMES)
    pitches        : (B,) MIDI pitches, dtype=torch.long
    n_steps        : Euler integration steps
    guidance_scale : float; 1.0 = no guidance, 3–7 = strong

    Returns
    -------
    (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    ### YOUR CODE HERE ###
    raise NotImplementedError

In [ ]:
# === Autograder Test 2b — do not modify ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_cfg1   = cfg_sample(model, x0_ag.clone(), p_ag, n_steps=20, guidance_scale=1.0)
out_euler  = euler_sample(model, x0_ag.clone(), p_ag, n_steps=20)
out_cfg6   = cfg_sample(model, x0_ag.clone(), p_ag, n_steps=20, guidance_scale=6.0)
out_naive2 = naive_scale_sample(model, x0_ag.clone(), p_ag, n_steps=20, scale=2.0)

assert torch.allclose(out_cfg1, out_euler, atol=1e-5), \
    'cfg_sample(guidance_scale=1.0) must equal euler_sample'
assert not torch.allclose(out_cfg6, out_euler, atol=1e-3), \
    'cfg_sample(guidance_scale=6.0) should differ from scale=1.0'
assert not torch.allclose(out_cfg6, out_naive2, atol=1e-3), \
    'CFG (scale=6) must differ from naive scaling (scale=2) — they use different formulas'
print('\u2713 cfg_sample | gs=1.0 matches Euler, gs=6.0 differs, CFG != naive scaling')

In [ ]:
# Listen: same noise, vary guidance scale 1 → 3 → 6 → 10
torch.manual_seed(42)
test_pitch = torch.full((1,), 60, dtype=torch.long, device=device)
test_noise = torch.randn(1, 2, FREQ_BINS, TIME_FRAMES, device=device)

for gs in [1.0, 3.0, 6.0, 10.0]:
    s = cfg_sample(model, test_noise.clone(), test_pitch, n_steps=50, guidance_scale=gs)
    audio = spec_to_audio(s[0].cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    print(f'guidance_scale={gs}  — pitch adherence increases, diversity decreases at high scales')
    display(ipd.Audio(audio.numpy(), rate=SR))

---
## Part 3a — Heun's Method  `[20 pts]`

Euler's method has **first-order** local truncation error $O(\Delta t^2)$.  
**Heun's method** (improved Euler / explicit trapezoidal rule) achieves **second-order** accuracy
$O(\Delta t^3)$ by using two velocity evaluations per step:

$$k_1 = \theta(x_t,\, t)$$
$$\hat{x} = x_t + k_1 \cdot \Delta t \quad\text{(Euler predictor)}$$
$$k_2 = \theta(\hat{x},\, t + \Delta t)$$
$$x_{t+\Delta t} = x_t + \tfrac{1}{2}(k_1 + k_2) \cdot \Delta t \quad\text{(corrected update)}$$

At the **same NFE budget** (number of function evaluations), Heun with $n/2$ steps
uses the same compute as Euler with $n$ steps, but with much lower discretization error.

**With CFG:** replace each $\theta$ evaluation with the CFG combination:
$$v_{\text{guided}} = v_{\text{uncond}} + s \cdot (v_{\text{cond}} - v_{\text{uncond}})$$

**Hint:** When `guidance_scale=1.0`, you only need one model call per velocity evaluation.

In [ ]:
def heun_sample(model, x0, pitches, n_steps=50, guidance_scale=1.0):
    """
    Heun's method (2nd-order Runge-Kutta) with optional CFG.

    Parameters: same as cfg_sample.
    Returns: (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    ### YOUR CODE HERE ###
    raise NotImplementedError

In [ ]:
# === Autograder Test 3a — do not modify ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_euler = euler_sample(model, x0_ag.clone(), p_ag, n_steps=25)
out_heun  = heun_sample( model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)

assert out_heun.shape == (4, 2, FREQ_BINS, TIME_FRAMES), \
    f'Wrong shape: {out_heun.shape}'
assert out_heun.isfinite().all(), 'Heun output contains NaN/Inf'
assert not torch.allclose(out_heun, out_euler, atol=1e-3), \
    'heun_sample must differ from euler_sample'

# CFG version: gs=1 should match no-CFG Heun
out_heun_gs1 = heun_sample(model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)
assert torch.allclose(out_heun, out_heun_gs1, atol=1e-5), \
    'heun_sample results should be deterministic given the same inputs'

out_heun_gs6 = heun_sample(model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=6.0)
assert not torch.allclose(out_heun_gs6, out_heun, atol=1e-3), \
    'heun_sample with guidance_scale=6 should differ from guidance_scale=1'

l2 = (out_heun - out_euler).norm().item()
print(f'\u2713 heun_sample | shape OK, differs from Euler (L2={l2:.3f}), CFG works')

---
## Part 3b — RK4  `[5 pts bonus]`

The **classic 4th-order Runge-Kutta** method uses 4 velocity evaluations per step:

$$k_1 = \theta(x_t,\;t)$$
$$k_2 = \theta(x_t + k_1\tfrac{\Delta t}{2},\;t + \tfrac{\Delta t}{2})$$
$$k_3 = \theta(x_t + k_2\tfrac{\Delta t}{2},\;t + \tfrac{\Delta t}{2})$$
$$k_4 = \theta(x_t + k_3\Delta t,\;t + \Delta t)$$
$$x_{t+\Delta t} = x_t + \tfrac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

**Note:** The half-step evaluations at $t + \Delta t/2$ are outside the training distribution
(which only sampled $t \sim U[0,1]$, not fractional sub-steps).
Observe whether this causes any quality difference vs. Heun at the same NFE.

In [ ]:
def rk4_sample(model, x0, pitches, n_steps=25, guidance_scale=1.0):
    """
    4th-order Runge-Kutta with optional CFG.  Bonus (5 pts).

    Parameters: same as heun_sample.
    Returns: (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    ### YOUR CODE HERE (BONUS) ###
    raise NotImplementedError

In [ ]:
# === Autograder Test 3b (bonus) — do not modify ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

try:
    out_rk4 = rk4_sample(model, x0_ag.clone(), p_ag, n_steps=12, guidance_scale=1.0)
    assert out_rk4.shape == (4, 2, FREQ_BINS, TIME_FRAMES)
    assert out_rk4.isfinite().all()
    out_heun = heun_sample(model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)
    assert not torch.allclose(out_rk4, out_heun, atol=1e-3), 'RK4 should differ from Heun'
    print('\u2713 rk4_sample (bonus): shape OK, differs from Heun')
except NotImplementedError:
    print('rk4_sample not implemented yet (bonus — skip if not attempting)')

In [ ]:
# Compare solvers at equal NFE budget (50 model evaluations each)
# Euler: 50 steps × 1 eval/step = 50 NFE
# Heun:  25 steps × 2 eval/step = 50 NFE   (+ CFG doubles each: 25 steps × 4 = 100 NFE)
# RK4:   12 steps × 4 eval/step = 48 NFE

torch.manual_seed(7)
pitch1 = torch.full((1,), 60, dtype=torch.long, device=device)
noise1 = torch.randn(1, 2, FREQ_BINS, TIME_FRAMES, device=device)

GS = 6.0  # guidance scale for all

comparisons = [
    ('Euler  (50 steps, gs=6)', lambda: cfg_sample(  model, noise1.clone(), pitch1, n_steps=50,  guidance_scale=GS)),
    ('Heun   (25 steps, gs=6)', lambda: heun_sample( model, noise1.clone(), pitch1, n_steps=25,  guidance_scale=GS)),
]
try:
    rk4_sample(model, noise1.clone(), pitch1, n_steps=1, guidance_scale=1.0)
    comparisons.append(('RK4    (12 steps, gs=6)', lambda: rk4_sample(model, noise1.clone(), pitch1, n_steps=12, guidance_scale=GS)))
except NotImplementedError:
    pass

for label, fn in comparisons:
    s = fn()
    audio = spec_to_audio(s[0].cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    print(label)
    display(ipd.Audio(audio.numpy(), rate=SR))

---
## Part 4 — Fine-tuning on a New Instrument  `[20 pts]`

The pretrained model only knows **keyboard** sounds. In this part you will:
1. Implement one gradient step of OT-CFM training (`train_step`).
2. Fine-tune the pretrained model on **guitar** (or another instrument of your choice).
3. Generate and submit 100 samples.

### The OT-CFM training step

Given a data batch $x_1$ and pitch labels $p$:

1. Sample $t \sim U[0,1]$ or $t = \sigma(\mathcal{N}(0,1))$ (logit-normal, concentrates near 0.5)
2. Sample $x_0 \sim \mathcal{N}(0,I)$  (fresh noise, same shape as $x_1$)
3. Interpolate: $x_t = (1-t)x_0 + t\,x_1$
4. Target velocity: $v^* = x_1 - x_0$
5. **CFG dropout:** with probability `p_uncond`, replace $p$ with `NULL_PITCH`
6. Predict: $\hat{v} = \theta(x_t, t, p)$
7. Loss: $\mathcal{L} = \|\hat{v} - v^*\|^2_F / (B \cdot 2 \cdot F \cdot T)$  (MSE)
8. `optimizer.zero_grad()` → `loss.backward()` → clip grad norm to 1.0 → `optimizer.step()`

**Hints:**
- `t` has shape `(B,)`; when multiplying with `x1` of shape `(B,2,F,T)`, broadcast via `t[:,None,None,None]`.
- logit-normal: `t = torch.sigmoid(torch.randn(B, device=x1.device))`
- `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` clips gradient norms.
- Return `loss.item()` (a Python float) so the training loop can log it.

In [ ]:
def train_step(model, optimizer, x1, pitch, p_uncond=0.1, t_sample='logit_normal'):
    """
    One OT-CFM gradient step.

    Parameters
    ----------
    model     : flow model  (should be in train mode when you call this)
    optimizer : a torch optimizer already constructed for model.parameters()
    x1        : (B, 2, FREQ_BINS, TIME_FRAMES)  data batch, on device
    pitch     : (B,) MIDI pitch labels, dtype=torch.long, on device
    p_uncond  : CFG dropout probability (replace pitch with NULL_PITCH)
    t_sample  : 'uniform' | 'logit_normal'

    Returns
    -------
    loss : float  (scalar loss value for logging)
    """
    ### YOUR CODE HERE ###
    raise NotImplementedError

In [ ]:
# === Autograder Test 4 — do not modify ===
import copy
torch.manual_seed(99)

# Synthetic batch (no data download needed for this test)
x1_ag  = torch.randn(8, 2, FREQ_BINS, TIME_FRAMES, device=device) * 0.5
p_ag   = torch.randint(0, 128, (8,), dtype=torch.long, device=device)

model_ag = copy.deepcopy(model)
opt_ag   = torch.optim.AdamW(model_ag.parameters(), lr=1e-4)
params_before = [p.data.clone() for p in model_ag.parameters()]

model_ag.train()
loss_val = train_step(model_ag, opt_ag, x1_ag, p_ag, p_uncond=0.1, t_sample='uniform')
model_ag.eval()

# Return type check
is_scalar = (torch.is_tensor(loss_val) and loss_val.ndim == 0) or isinstance(loss_val, float)
assert is_scalar, f'train_step must return a scalar, got {type(loss_val)}'
loss_float = float(loss_val.item() if torch.is_tensor(loss_val) else loss_val)

# Range check
assert 0 < loss_float < 10, f'Loss {loss_float:.4f} outside expected range (0, 10)'

# Parameters must change
changed = any(not torch.allclose(pb, pa)
              for pb, pa in zip(params_before, model_ag.parameters()))
assert changed, 'Model parameters did not change — did you call optimizer.step()?'

print(f'\u2713 train_step | loss={loss_float:.4f}, parameters updated')

### 4b — Download fine-tuning data

We use `nsynth-valid` (~1.4 GB), which contains all instrument families.  
Use `instrument_filter` to select your target (default: `'guitar'`).  
Available families: `bass, brass, flute, guitar, keyboard, mallet, organ, reed, string, synth_lead, vocal`

In [ ]:
DATA_ROOT = '/content/nsynth'
VALID_DIR = f'{DATA_ROOT}/nsynth-valid/audio'

if not os.path.exists(VALID_DIR):
    print('Downloading nsynth-valid (~1.4 GB)...')
    !mkdir -p {DATA_ROOT}
    !wget -q http://download.magenta.tensorflow.org/datasets/nsynth/nsynth-valid.jsonwav.tar.gz \
           -O /tmp/nsynth-valid.tar.gz
    !tar -xf /tmp/nsynth-valid.tar.gz -C {DATA_ROOT}
    !rm /tmp/nsynth-valid.tar.gz
    print('Done.')
else:
    print('nsynth-valid already present.')

import glob
all_valid = glob.glob(f'{VALID_DIR}/*.wav')
print(f'Total valid files: {len(all_valid):,}')
for family in ['guitar', 'bass', 'flute', 'brass', 'reed', 'keyboard']:
    n = len([f for f in all_valid if os.path.basename(f).startswith(family)])
    print(f'  {family:12s}: {n:4d} files')

In [ ]:
# ── YOUR CHOICES ──────────────────────────────────────────────────────────────
TARGET_INSTRUMENT = 'guitar'    # Change to any family listed above
FT_MAX_FILES      = 2000        # Number of files to use
FT_EPOCHS         = 300         # Training epochs
FT_LR             = 1e-3        # Learning rate
FT_BATCH_SIZE     = 64
# ─────────────────────────────────────────────────────────────────────────────

import copy, time
from tqdm.notebook import trange

# Fresh copy of pretrained weights
ckpt_ft  = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model_ft = build_model_from_config(ckpt_ft['config']).to(device)
model_ft.load_state_dict(ckpt_ft['model_state'])

dataset_ft = NSynthSpecDataset(VALID_DIR, instrument_filter=TARGET_INSTRUMENT,
                               max_files=FT_MAX_FILES)
loader_ft  = DataLoader(dataset_ft, batch_size=FT_BATCH_SIZE, shuffle=True,
                        drop_last=True, num_workers=2)
optimizer_ft = torch.optim.AdamW(model_ft.parameters(), lr=FT_LR, weight_decay=1e-4)

print(f'Fine-tuning on {len(dataset_ft)} {TARGET_INSTRUMENT} files, '
      f'{len(loader_ft)} batches/epoch, {FT_EPOCHS} epochs')

t0 = time.time()
for epoch in range(1, FT_EPOCHS + 1):
    model_ft.train()
    epoch_loss = []
    for x1, pitch in loader_ft:
        x1    = x1.to(device, non_blocking=True)
        pitch = pitch.to(device, non_blocking=True)
        loss  = train_step(model_ft, optimizer_ft, x1, pitch, p_uncond=0.1)
        epoch_loss.append(float(loss))
    if epoch % max(1, FT_EPOCHS // 10) == 0 or epoch == 1:
        print(f'Epoch {epoch:4d}/{FT_EPOCHS}  loss={np.mean(epoch_loss):.4f}  '
              f'elapsed={time.time()-t0:.0f}s')

model_ft.eval()
print(f'\nFine-tuning done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Save fine-tuned checkpoint (needed for submission reproducibility check)
import copy as _copy
FT_CKPT_PATH = '/content/model_ft_q4.pt'
torch.save({
    'model_state': model_ft.state_dict(),
    'config':      ckpt_ft['config'],
    'n_params':    ckpt_ft['n_params'],
    'instrument':  TARGET_INSTRUMENT,
}, FT_CKPT_PATH)
print(f'Checkpoint saved to {FT_CKPT_PATH}')

### 4c — Generate & submit 100 samples

Choose your best sampler and guidance scale.
**The submission must include the starting noise for each sample** so we can verify reproducibility.

In [ ]:
# ── YOUR CHOICES ──────────────────────────────────────────────────────────────
Q4_SAMPLER        = 'heun'   # 'euler' | 'cfg' | 'heun' | 'rk4'
Q4_GUIDANCE_SCALE = 6.0
Q4_N_STEPS        = 50
# ─────────────────────────────────────────────────────────────────────────────

N_SUB = 100
# Spread pitches across 3 octaves (C3–B5)
q4_pitches = torch.tensor(
    [(48 + i % 36) for i in range(N_SUB)], dtype=torch.long, device=device)

torch.manual_seed(0)
q4_noises = torch.randn(N_SUB, 2, FREQ_BINS, TIME_FRAMES, device=device)

model_ft.eval()
q4_samples = []
BATCH = 16

with torch.no_grad():
    for i in range(0, N_SUB, BATCH):
        x0b = q4_noises[i:i+BATCH]
        pb  = q4_pitches[i:i+BATCH]
        if Q4_SAMPLER == 'euler':
            out = euler_sample(model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS)
        elif Q4_SAMPLER == 'heun':
            out = heun_sample( model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS,
                               guidance_scale=Q4_GUIDANCE_SCALE)
        elif Q4_SAMPLER == 'rk4':
            out = rk4_sample(  model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS,
                               guidance_scale=Q4_GUIDANCE_SCALE)
        else:  # cfg
            out = cfg_sample(  model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS,
                               guidance_scale=Q4_GUIDANCE_SCALE)
        q4_samples.append(out.cpu())

q4_samples = torch.cat(q4_samples)  # (100, 2, F, T)
print(f'Generated {len(q4_samples)} samples')

# Listen to a few
for i in range(3):
    audio = spec_to_audio(q4_samples[i])
    audio = audio / (audio.abs().max() + 1e-8)
    print(f'Sample {i}, pitch={q4_pitches[i].item()}')
    display(ipd.Audio(audio.numpy(), rate=SR))

In [ ]:
# Save submission for Q4
os.makedirs('/content/submission_q4', exist_ok=True)

np.savez_compressed(
    '/content/submission_q4/submission.npz',
    samples        = q4_samples.numpy().astype(np.float32),
    noises         = q4_noises.cpu().numpy().astype(np.float32),
    pitches        = q4_pitches.cpu().numpy().astype(np.int64),
    guidance_scale = np.array(Q4_GUIDANCE_SCALE, dtype=np.float32),
    n_steps        = np.array(Q4_N_STEPS,        dtype=np.int32),
    sampler        = np.array(Q4_SAMPLER,         dtype=object),
)
print('Saved: /content/submission_q4/submission.npz')
print()
print('Submit the following files:')
print('  1. /content/submission_q4/submission.npz')
print('  2. /content/model_ft_q4.pt  (your fine-tuned checkpoint)')
print('  3. This notebook (.ipynb)')

---
## Part 5 — Beat the Baseline  `[10 pts]`

**Baseline** (pretrained keyboard model, Heun 25 steps, guidance=6):  
- FD@6 ≈ 354  
- Pitch class accuracy ≈ 79%

Achieve a **lower FD** or **higher pitch accuracy** using any approach:

| Idea | Notes |
|---|---|
| More fine-tuning epochs | Longer training on same data |
| Different target instrument | Easier task = better FD |
| Better inference | Heun/RK4 vs. Euler; higher guidance scale |
| Mixed training | Fine-tune on multiple families |
| Train from scratch | Use `train_step` with a fresh model |
| Model architecture | Load a larger model config |

Describe your approach in the markdown cell below, then generate and save 100 samples.

### My approach for Part 5

*(Replace this with your description: what did you try, what worked, and why?)*

In [ ]:
# ── YOUR CODE HERE — no structure imposed ─────────────────────────────────────
# Train a better model (model_q5), then generate q5_samples and q5_noises below.

# Example starting point — fine-tune longer on keyboard (the easiest task):
# ckpt_q5  = torch.load(CKPT_PATH, map_location=device, weights_only=False)
# model_q5 = build_model_from_config(ckpt_q5['config']).to(device)
# model_q5.load_state_dict(ckpt_q5['model_state'])
# ... your training loop ...

In [ ]:
# Generate 100 samples from your best Q5 model
# ── YOUR CHOICES ──────────────────────────────────────────────────────────────
Q5_SAMPLER        = 'heun'
Q5_GUIDANCE_SCALE = 6.0
Q5_N_STEPS        = 50
# Use model_q5 (defined in cell above)
# ─────────────────────────────────────────────────────────────────────────────

q5_pitches = torch.tensor(
    [(48 + i % 36) for i in range(N_SUB)], dtype=torch.long, device=device)

torch.manual_seed(0)
q5_noises = torch.randn(N_SUB, 2, FREQ_BINS, TIME_FRAMES, device=device)

# model_q5.eval()
# q5_samples = ... (same pattern as the Q4 generation cell)

# Once generated, run the save cell below:
print('TODO: generate q5_samples (100, 2, FREQ_BINS, TIME_FRAMES) and q5_noises')

In [ ]:
# Save submission for Q5
os.makedirs('/content/submission_q5', exist_ok=True)

Q5_CKPT_PATH = '/content/model_q5.pt'
# torch.save({'model_state': model_q5.state_dict(), 'config': ..., 'n_params': ...}, Q5_CKPT_PATH)

np.savez_compressed(
    '/content/submission_q5/submission.npz',
    samples        = q5_samples.numpy().astype(np.float32),   # fill in q5_samples
    noises         = q5_noises.cpu().numpy().astype(np.float32),
    pitches        = q5_pitches.cpu().numpy().astype(np.int64),
    guidance_scale = np.array(Q5_GUIDANCE_SCALE, dtype=np.float32),
    n_steps        = np.array(Q5_N_STEPS,        dtype=np.int32),
    sampler        = np.array(Q5_SAMPLER,         dtype=object),
)
print('Saved: /content/submission_q5/submission.npz')
print()
print('Submit:')
print('  1. /content/submission_q5/submission.npz')
print('  2. /content/model_q5.pt')
print('  3. This notebook (.ipynb)')

---
## Submission checklist

Before submitting, run **Kernel → Restart and Run All** to confirm everything executes
from scratch without errors.

| Item | File | Required for |
|---|---|---|
| Completed notebook | `assignment.ipynb` | All parts |
| Q4 samples + noises | `submission_q4/submission.npz` | Part 4 |
| Q4 checkpoint | `model_ft_q4.pt` | Part 4 |
| Q5 samples + noises | `submission_q5/submission.npz` | Part 5 |
| Q5 checkpoint | `model_q5.pt` | Part 5 |

**How the autograder verifies Q4/Q5:**  
It will load your checkpoint, call your sampler function on a random subset of the
submitted noises with the submitted hyperparameters, and compare the outputs to your
submitted samples. Make sure your inference functions are deterministic given the same
inputs (no extra internal randomness).